# 8. Event Matching & Match Evaluation

Sample-level metrics (notebook 5) compare label sequences point by point; they can't tell you whether a detector
found "the same events" as a human rater, just shifted in time. `peyes.match` pairs up events from two sequences
based on a chosen criterion, and `peyes.match_metrics` then measures how well the paired-up events agree.

In [1]:
import numpy as np
import peyes
import _helpers

d = _helpers.load_example_trial()
ground_truth = peyes.create_events(
    labels=d["raters"]["RA"], t=d["t"], x=d["x"], y=d["y"], pupil=d["pupil"],
    pixel_size=d["pixel_size"], viewer_distance=d["viewer_distance"],
)
detector = peyes.create_detector("engbert", missing_value=np.nan, min_event_duration=4, pad_blinks_time=0)
labels, _ = detector.detect(
    t=d["t"], x=d["x"], y=d["y"], pixel_size_cm=d["pixel_size"], viewer_distance_cm=d["viewer_distance"],
)
prediction = peyes.create_events(
    labels=labels, t=d["t"], x=d["x"], y=d["y"], pupil=d["pupil"],
    pixel_size=d["pixel_size"], viewer_distance=d["viewer_distance"],
)
len(ground_truth), len(prediction)

(43, 62)

## Matching strategies

`peyes.match(ground_truth, prediction, match_by, **kwargs)` maps each ground-truth event to the "best" predicted
event under the chosen criterion:

| `match_by` | pairs by | key kwarg |
|---|---|---|
| `"iou"` | maximum time-intersection-over-union | `min_iou` |
| `"max"` | maximum (duration-normalized) overlap | `min_overlap` |
| `"first"` / `"last"` | first/last overlapping event | `min_overlap` |
| `"onset"` / `"offset"` | least onset/offset time difference | `max_onset_difference` / `max_offset_difference` |
| `"window"` | onset *and* offset within a window | both of the above |
| `"l2"` | minimum L2 norm of (onset diff, offset diff) | `max_l2` |

Events with no acceptable match under the criterion are simply left out of the result. Comparing a few strategies:

In [2]:
strategies = {
    "iou (>=0.3)": dict(match_by="iou", min_iou=0.3),
    "onset (<=15ms)": dict(match_by="onset", max_onset_difference=15),
    "window (<=15ms)": dict(match_by="window", max_onset_difference=15, max_offset_difference=15),
}
for name, kwargs in strategies.items():
    matches = peyes.match(ground_truth, prediction, **kwargs)
    print(f"{name}: {len(matches)}/{len(ground_truth)} ground-truth events matched")

iou (>=0.3): 27/43 ground-truth events matched
onset (<=15ms): 26/43 ground-truth events matched
window (<=15ms): 21/43 ground-truth events matched


By default, an event can only match another event of the *same* label; pass `allow_xmatch=True` to allow
matching across labels (e.g. a ground-truth smooth-pursuit event matched to a predicted fixation).

## Evaluating the matches

`peyes.match_metrics` takes the `matches` dict from `peyes.match` (plus the original sequences where needed) and
computes per-pair differences or aggregate scores. Element-wise feature differences, one value per matched pair:

In [3]:
matches = peyes.match(ground_truth, prediction, match_by="iou", min_iou=0.3)
print("duration differences (ms):", peyes.match_metrics.duration_difference(matches)[:5])
print("center-pixel distances (px):", peyes.match_metrics.center_pixel_distance(matches)[:5])
print("time IoU:", peyes.match_metrics.time_iou(matches)[:5])

duration differences (ms): [   6.     -24.01  -132.023   -4.      -7.998]
center-pixel distances (px): [ 0.13502009 38.17644481 10.45729809  0.58888879  3.71245632]
time IoU: [0.97143483 0.59990002 0.40541965 0.83338887 0.73347996]


Aggregate, event-level scores — treating a match as a "hit" the same way sample-level metrics treat a
correct label:

In [4]:
fixation = peyes.parse_label("fixation")
print(f"match ratio: {peyes.match_metrics.match_ratio(prediction, matches):.2%}")
precision, recall, f1 = peyes.match_metrics.precision_recall_f1(ground_truth, prediction, matches, positive_label=fixation)
print(f"precision={precision:.2f}  recall={recall:.2f}  f1={f1:.2f}")
print(f"false alarm rate: {peyes.match_metrics.false_alarm_rate(ground_truth, prediction, matches, positive_label=fixation):.2%}")
d_prime, criterion = peyes.match_metrics.d_prime_and_criterion(ground_truth, prediction, matches, positive_label=fixation)
print(f"d-prime={d_prime:.2f}  criterion={criterion:.2f}")

match ratio: 43.55%
precision=0.37  recall=1.00  f1=0.54
false alarm rate: 59.38%
d-prime=1.78  criterion=-1.12


## What's next

**[9 Temporal Alignment Evaluation](./9%20Temporal%20Alignment%20Evaluation.ipynb)** — a complementary, sample-level
view of onset/offset timing accuracy, without needing an explicit event-to-event match.